# Module 38.2 — Dataset Engineering

**Jupyter Notebook Notes · Computer Vision · AI/ML Engineer Roadmap**

---

## 1. What is Dataset Engineering?

Dataset engineering is the process of preparing, organising, validating, and maintaining data before using it to train a machine learning model.

In image classification, the quality of your dataset directly affects how well your model learns and performs on unseen images.

### Practical Example

Imagine building a model to identify defective motor components.

You collect 10,000 images, but:

- 8,000 images show normal components.
- Only 2,000 images show defective components.
- Some images are duplicated.
- Some images are incorrectly labelled.
- Some images from the test set are also present in training.

Even if you build a powerful deep learning model, these problems can produce misleading results.

Dataset engineering ensures that the model learns from reliable, representative, and correctly organised data.

### Visual References

![Dataset engineering and image classification](https://images.openai.com/static-rsc-4/LRd81X3P4hocWAsFWgaKoOsHMyRiCXSW-NY91e69vOSTMI-5zIspGZ-_Xpm9kEXXn6Uifjw6rmSpYCGOSW0ELVrzgs8uwu8Z92cw-2O9DUMSM9zShHSZxBq0c2fC3dSFHvwj7SXMyM8Mce0q2QoyeuIzIPPw9tsb-tXy2ZYKD7M?purpose=inline)

![Industrial defect detection](https://images.openai.com/static-rsc-4/bhlkre3TiRgBUNM8VbZ59BXyNB5Qmj2r1Eqdr4boOWZ_XO9zgzMpct_-sTvRrZoCr7k_7HO0mKEZkReLJMGRGsJVjNwtRAQarS8DUOT8Gk55PgG8YaVy8ehRZq4CLBYOM00Cq0zNth_2k9h71BTGzIaaieEu02Ufm-WGuzo5CnE?purpose=inline)

---

## 2. Train / Validation / Test Split

A dataset is generally divided into three subsets.

| Dataset | Purpose |
|---|---|
| Training | Teaches the model by updating its parameters |
| Validation | Helps monitor performance and make development decisions |
| Testing | Evaluates the final model on held-out data |

### Typical Dataset Distribution

A common starting point is:

- Training: 70%
- Validation: 15%
- Testing: 15%

### A. Training Dataset

The training dataset is used to teach the model.

During training:

1. Images are passed into the model.
2. The model generates predictions.
3. Loss is calculated.
4. Backpropagation calculates gradients.
5. The optimizer updates model parameters.

Example:

```text
Training images: 7,000

Normal       → 4,000
Defective    → 3,000
```

### B. Validation Dataset

The validation dataset helps evaluate the model during development.

It is useful for:

- Comparing model architectures.
- Selecting hyperparameters.
- Monitoring overfitting.
- Selecting checkpoints.
- Choosing training duration.

**Important:** Validation images are not used for gradient-based parameter updates.

### C. Test Dataset

The test dataset is reserved for final evaluation.

It helps estimate how well the trained model performs on previously unseen data.

**Important:** Do not repeatedly use test results to tune the model. Doing so indirectly introduces test-set information into model development.

### Practical Example

Suppose you have 10,000 images:

| Split | Percentage | Images |
|---|---:|---:|
| Training | 70% | 7,000 |
| Validation | 15% | 1,500 |
| Testing | 15% | 1,500 |
| **Total** | **100%** | **10,000** |

### PyTorch Implementation

```python
from torch.utils.data import random_split

dataset = YourImageDataset()

train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size]
)
```

This is a basic random split. It does not guarantee that every class has the same proportion in each subset.

That brings us to stratification.

---

## 3. Stratification

Stratification is a dataset-splitting technique that preserves approximately the same class distribution across training, validation, and test subsets.

### Why is Stratification Important?

Consider a dataset containing 1,000 images:

| Class | Total Images | Percentage |
|---|---:|---:|
| Normal | 800 | 80% |
| Defective | 200 | 20% |
| **Total** | **1,000** | **100%** |

Suppose you randomly split the dataset without stratification.

You might get:

| Class | Training | Validation | Test |
|---|---:|---:|---:|
| Normal | 550 | 130 | 120 |
| Defective | 150 | 20 | 30 |

The class proportions are not exactly preserved.

This can become more problematic when a dataset has very few examples of a particular class. A validation or test set might contain too few examples to evaluate that class reliably.

### Stratified Splitting

With stratification, the approximate 80:20 class ratio is preserved in each subset.

### Visual Reference

![Stratified train test split](https://images.openai.com/static-rsc-4/_jYyIX2SadMW-84mCSGa1MVKZSRHnZvEHnzmV2TQ5t2YkedMp1KFPpSJKULYiw_tKGcdWaU_iPR0f8yt3yzRbqlgBXdS3aamt8w-wAEHZ2JKLA6G6j0Zlj4EhAXSTMUpJVM8kGsgscpxyn8Q0kzTgqGrQDBMqlTtXUf_CDO4Zic?purpose=inline)

### Example Using Scikit-learn

```python
from sklearn.model_selection import train_test_split

# X = image paths
# y = corresponding class labels

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)
```

This produces approximately:

- 70% training
- 15% validation
- 15% testing

**Important:** Stratification preserves class proportions, but it does not automatically prevent duplicate images or data leakage.

**When to use it:** Particularly useful when classes are unevenly represented or the dataset is relatively small.

---

## 4. Class Imbalance

Class imbalance occurs when some classes have significantly more samples than others.

### Example

Imagine a manufacturing defect classification dataset:

| Class | Number of Images |
|---|---:|
| Normal | 9,000 |
| Surface defect | 700 |
| Crack | 200 |
| Deformation | 100 |

![Class imbalance visualization](https://images.openai.com/static-rsc-4/8FBZ1Ehn5k2Dm8rASiLOQA-LOIWAzsy8PNLkHZ8u7JrZI0PX-dLLynI88S-6YzxB3nA7Tuhi17cs37yikNEtLexJfwDA4TQRcorPo5tLTUvZ-1e2JM1gYVx3a_sP4vnd3vPbjgpBNPySNox5ErYkltI4ELlbTq7m6DIaJ4AQikI?purpose=inline)

The normal class represents 90% of the dataset.

A model could predict `Normal` for every image and achieve 90% accuracy, despite completely failing to recognise defects.

This is why accuracy alone can be misleading.

### Techniques to Handle Class Imbalance

### A. Oversampling

Increase the representation of minority classes by sampling their examples more frequently.

Example:

```text
Before:

Normal       → 9000
Defective    → 1000

After oversampling:

Normal       → 9000
Defective    → 9000
```

Oversampling can increase exposure to minority examples, but simply repeating images may cause overfitting.

### B. Data Augmentation

Generate modified training examples using transformations such as:

- Rotation.
- Horizontal flipping.
- Cropping.
- Brightness changes.
- Contrast adjustments.

![Image augmentation examples](https://images.openai.com/static-rsc-4/xNdI2nETzxNCpFZLf95b7nqCDc5AQDU5vWp5Kh3fpX0cyx_RKr7aOh9J5DxWoKooxUJtVeqhZ2TO6wjxvW3NlXR36H9aOCczPtI6FWYw5vXA8e4SHfAo89SLVrV77I6q-c8fi3Re2C6DtvV-Djz69j22_W00F_gIZFLUydZznF8?purpose=inline)

Example:

```python
from torchvision import transforms

augmentation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor()
])
```

Use only transformations that preserve the meaning of the image.

For example, flipping may be inappropriate if orientation carries important information.

### C. Undersampling

Reduce the number of majority-class samples.

Example:

```text
Before:

Normal       → 9000
Defective    → 1000

After:

Normal       → 1000
Defective    → 1000
```

**Advantage:** Reduces training time and balances the dataset.

**Disadvantage:** You may discard useful information from the majority class.

### D. Class-Weighted Loss

Assign higher loss weights to minority classes so that mistakes on those classes contribute more strongly to the training objective.

Example:

```python
import torch
import torch.nn as nn

class_weights = torch.tensor(
    [1.0, 3.0, 5.0, 8.0],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)
```

The weights above are illustrative. In practice, calculate or tune them based on the training distribution and validation performance.

### Comparison

| Technique | Main Idea |
|---|---|
| Oversampling | Increase minority-class representation |
| Augmentation | Create varied training examples |
| Undersampling | Reduce majority-class representation |
| Class-weighted loss | Penalise minority-class mistakes more |

**Engineering Recommendation:** Start by examining the class distribution. Then consider class-weighted loss and suitable augmentation before discarding large amounts of data.

---

## 5. Duplicate Detection

Duplicate detection identifies repeated images in a dataset.

Duplicates can occur when:

- Images are downloaded multiple times.
- The same image is stored under different filenames.
- A dataset is collected from multiple sources.
- Images are resized or compressed and saved again.

### Why Are Duplicates a Problem?

Suppose an identical image appears in both training and testing.

The model has already seen that image during training. It may perform well on the test set because it recognises familiar content rather than genuinely generalising.

![Image deduplication](https://images.openai.com/static-rsc-4/GsaDh400tGgaAKD1d8kBUXlaHse861-5oPRf0rSa2lYG7o7PRu-KaZ47cIOTTZMPvOeQh_RaLTy8HooUtDq5NyX95JPQxOPcR__6G6hmY1-Goma-8iptjPkeKx6kVeyzhFWNRqaJceXg6copaaAxuKbW62_8khZafcKtl_pywl4?purpose=inline)

### A. Exact Duplicate Detection

An exact duplicate has identical file content.

A cryptographic hash such as SHA-256 can identify identical files.

```python
import hashlib

def calculate_hash(image_path):
    with open(image_path, "rb") as file:
        return hashlib.sha256(file.read()).hexdigest()
```

If two files produce the same SHA-256 hash, they have identical contents, barring an extraordinarily unlikely hash collision.

### B. Near-Duplicate Detection

Near-duplicates are visually similar images that may differ in:

- Resolution.
- Compression.
- Brightness.
- Cropping.
- Small edits.

For this purpose, perceptual hashing (pHash) is useful.

![Perceptual image similarity](https://images.openai.com/static-rsc-4/wHg3PdIN02Ku6dtnE2_K9ADt-qfy8mGXIppKcN66__9JlLb2zgzxlDntd_KSFCRck9DLwYoCXBGG5RLPut_IoCCkBiTlZ9M40iPbpzMpnSfDuKacnVW1aEehe1G5eo76hn4yeTBDhJ0o5CIIjmOpikA6oHXza7rVdDe9Ij35aLQ?purpose=inline)

Example installation:

```python
pip install pillow imagehash
```

Example:

```python
from PIL import Image
import imagehash

image1 = Image.open("image1.jpg")
image2 = Image.open("image2.jpg")

hash1 = imagehash.phash(image1)
hash2 = imagehash.phash(image2)

distance = hash1 - hash2

print("Hash distance:", distance)
```

A smaller Hamming distance indicates greater perceptual similarity. The threshold must be selected and validated for your dataset.

### Exact Hash vs. Perceptual Hash

| Feature | SHA-256 | pHash |
|---|---|---|
| Identifies exact duplicates | Yes | Usually |
| Detects resized copies | No | Often |
| Detects compressed copies | No | Often |
| Detects visually similar images | No | To some extent |

**Important:** Duplicate detection should happen before splitting the dataset, or at least before finalising the splits. Keep related images together rather than allowing near-identical examples to appear across training and testing.

---

## 6. Data Leakage Prevention

Data leakage occurs when information that should not be available during model training or development influences the training process or evaluation.

It can make a model appear much better during testing than it performs in the real world.

### Example: Image Leakage

Suppose you have 100 photographs of the same motor component, taken from slightly different angles.

You randomly split them:

- Training: 70 photographs.
- Validation: 15 photographs.
- Testing: 15 photographs.

Although the photographs are different, all three subsets contain images of the same physical component.

The model may learn component-specific visual characteristics. Consequently, the test score may not represent performance on entirely new components.

![Data leakage illustration](https://images.openai.com/static-rsc-4/fasqInmqD_dH74Pj08hHwnTdTuOUgXA4yG3Mbl9_ZnBpgR66EsAQW2uCC4QH1HKUUgAw2hobCnU6XiYT-FFYTlb86BJNENcfErlZosiO4Ue3Ld3jZhXnt13HRjYeN0fph0zfPtuQk0emX0h4OV6BK-vs9C11X6xIW1l7TCvTiXs?purpose=inline)

### Common Types of Data Leakage

**A. Duplicate Leakage**

The same image or a near-identical copy appears in multiple subsets.

**B. Group Leakage**

Images belonging to the same physical object, patient, production batch, video, or source are split across training and testing.

**C. Preprocessing Leakage**

Statistics such as mean and standard deviation are calculated using the entire dataset before splitting.

For example, calculating normalization statistics using training, validation, and test images together can introduce information from the held-out data.

**D. Test-Set Leakage**

The developer repeatedly checks test results and adjusts model architecture, hyperparameters, or preprocessing based on those results.

The test set gradually becomes part of the development process.

### How to Prevent Leakage

| Problem | Prevention |
|---|---|
| Duplicate images | Detect duplicates before splitting |
| Same object in multiple splits | Use group-based splitting |
| Test information influencing training | Keep test data untouched during development |
| Preprocessing statistics leakage | Fit preprocessing only on training data |
| Related images across splits | Split by source, object, patient, or production batch |

### Group-Based Splitting

When multiple images belong to the same physical object, keep all images of that object in the same subset.

Scikit-learn provides `GroupShuffleSplit` for this purpose.

```python
from sklearn.model_selection import GroupShuffleSplit

# X = image paths
# y = labels
# groups = component IDs

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = [X[i] for i in train_idx]
X_test = [X[i] for i in test_idx]
```

Here, `groups` contains the identity of the physical component associated with each image.

For example:

```text
Image          Component ID
motor_01.jpg   Motor_A
motor_02.jpg   Motor_A
motor_03.jpg   Motor_B
motor_04.jpg   Motor_B
```

All images belonging to `Motor_A` must remain together.

**Must Remember:**

A random split separates images. A group-based split separates independent sources or objects.

Choose the split according to how the model will be used in the real world.

---

## 7. Annotation Quality

Annotation is the process of assigning correct labels or other relevant information to data.

In image classification, annotation generally means assigning the correct class label to each image.

### Example

Imagine these images belong to a motor-component dataset:

| Image | Correct Annotation |
|---|---|
| Image 1 | Normal |
| Image 2 | Crack |
| Image 3 | Surface defect |
| Image 4 | Deformation |

### What Happens When Labels Are Incorrect?

Suppose an image clearly contains a crack but is labelled as `Normal`.

During training:

1. The model processes the image.
2. It predicts a class.
3. The loss function compares the prediction against the incorrect label.
4. The model receives an incorrect training signal.

Repeated label errors can reduce model performance and make learning more difficult.

### Common Annotation Problems

**A. Incorrect Labels**

An image of a defective component is labelled as normal.

**B. Ambiguous Labels**

An image contains a defect, but its category is unclear.

**C. Inconsistent Labelling**

Two annotators assign different classes to visually similar images.

**D. Missing Labels**

Some images have no assigned class.

**E. Incorrect Class Definitions**

Two classes overlap or are poorly defined.

### Annotation Quality Improvement

1. Establish clear definitions for every class.
2. Create annotation guidelines with example images.
3. Review a sample of labelled images manually.
4. Use multiple reviewers for ambiguous or critical examples.
5. Resolve disagreements using a consistent review process.
6. Maintain a record of corrected annotations.

### Annotation Quality in Industry

In manufacturing, annotation guidelines should specify exactly what counts as:

- A surface scratch.
- A crack.
- Corrosion.
- Deformation.
- An acceptable component.

![Industrial surface defect inspection](https://images.openai.com/static-rsc-4/Z6KR7Oqf2JrXD79m4NKG-0txCuGdnetQE0L_7pBUKb4AzEHKZZ-y7MeE8CfxJx2ac0MUbjY34SIAQ_HkasSHxnI0gfT768sWdoYHbwbqikrN1BSqPnn78P5BRFJFXxAQX8RttdsK2y0QZycD29pDGY6WehWfeTJV6X3CKEiLp6E?purpose=inline)

A well-designed dataset needs consistent class definitions, not just a large number of images.

---

## 8. Complete Dataset Engineering Workflow

Let's connect all six concepts into one practical workflow.

For a real-world image classification project:

1. **Data Collection:** Collect images from cameras, databases, or public datasets.
2. **Annotation:** Assign correct class labels.
3. **Data Validation:** Check corrupted files, missing labels, and incorrect formats.
4. **Duplicate Detection:** Identify exact and near-duplicate images.
5. **Leakage Prevention:** Identify related images and group them appropriately.
6. **Dataset Splitting:** Create training, validation, and test sets.
7. **Stratification:** Preserve class proportions where appropriate.
8. **Class Imbalance Handling:** Apply augmentation, sampling, or weighted loss when needed.
9. **Final Verification:** Confirm that each split is suitable for its intended purpose.

### Recommended Dataset Folder Structure

```text
motor_defect_dataset/
│
├── train/
│   ├── normal/
│   ├── crack/
│   ├── surface_defect/
│   └── deformation/
│
├── val/
│   ├── normal/
│   ├── crack/
│   ├── surface_defect/
│   └── deformation/
│
├── test/
│   ├── normal/
│   ├── crack/
│   ├── surface_defect/
│   └── deformation/
│
└── dataset_metadata.csv
```

The metadata file can contain:

| image_path | label | component_id | batch_id |
|---|---|---|---|
| img001.jpg | normal | M001 | B01 |
| img002.jpg | crack | M002 | B01 |
| img003.jpg | normal | M003 | B02 |

This metadata helps with group splitting, traceability, and dataset auditing.

---

## 9. Must Know / Good to Know / Advanced

| Priority | Concepts |
|---|---|
| Must Know | Train/validation/test, stratification, class imbalance, duplicate detection, leakage prevention, annotation quality |
| Good to Know | Oversampling, undersampling, weighted loss, perceptual hashing, group-based splitting |
| Advanced | Automated data-quality pipelines, embedding-based duplicate detection, annotation agreement measurement, dataset versioning |

---

## 10. What You Should Remember 🧠

1. Training data teaches the model; validation data supports development; test data measures final generalisation.
2. Stratification preserves class proportions across dataset splits.
3. Class imbalance can make accuracy misleading.
4. Oversampling, augmentation, undersampling, and weighted loss are ways to address imbalance.
5. SHA-256 detects exact duplicates; perceptual hashing helps detect visually similar images.
6. Data leakage creates misleading evaluation results. Group-related images must stay together when appropriate.
7. Incorrect annotations teach the model incorrect relationships.

### Central Engineering Principle

> A powerful model trained on a poorly engineered dataset can perform worse than a simpler model trained on a reliable dataset.

---